In [1]:
!pip install anthropic pillow


In [2]:
import anthropic, base64, os, json, time
print(f"anthropic: {anthropic.__version__}")
print("Imports OK")


anthropic: 0.96.0
Imports OK


# NeurIPS Computational Resources & Reproducibility Metadata

Required by **NeurIPS 2025 Paper Checklist §8**. Run to print a live hardware/version summary. Fill `[TODO]` fields before submission.

In [3]:
"""
NeurIPS 2025 Checklist S8 - Computational Resources
====================================================
Hardware
  GPU      : NVIDIA RTX A6000 (48 GB VRAM)
  CPU      : [TODO: e.g. AMD EPYC 7542 32-core]
  RAM      : [TODO: e.g. 256 GB DDR4]
  OS       : Windows 11 / Ubuntu 22.04
  Provider : Local on-premise workstation

Model & Inference
  Model      : claude-opus-4-7
  max_tokens : 4096 per call
  Temp       : 0.0 (Zero-Shot / Sequential / LtM / ReAct / CoT)
               0.1 (Iterative)
               0.1-0.5 (Self-Consistency runs)
               0.1/0.7 (Meta-Prompting: analysis/generation)

API Calls per Video  (C = ceil(frames / 10))
  Zero-Shot        : C
  Sequential       : 5*C + 1
  Least-To-Most    : 8*C + 1
  ReAct            : C + 1
  True Iterative   : up to 8*C
  Self-Consistency : 5*C + 1
  Meta-Prompting   : 2*C + 2
  Chain-of-Thought : C + 1
  Total/video (50 frames, C=5) ~ 162 calls ~ 243 000 tokens

Total Compute (fill before submission)
  Dataset  : [TODO] videos x [TODO] avg frames
  Calls    : [TODO] x 162 = [TODO]
  Tokens   : [TODO] x 243 000 = [TODO]
  Time     : ~[TODO] hours

Reproducibility
  - Checkpoint files save progress after every video (atomic write)
  - Re-running any cell after failure resumes with zero extra API cost
  - All raw API responses saved as JSON before post-processing
  - Chunk-level saves after every 10-frame batch
"""
import subprocess, platform, datetime
print("=" * 64)
print(f"  NeurIPS Compute Summary  - {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("=" * 64)
print(f"  Python  : {platform.python_version()}")
print(f"  OS      : {platform.platform()}")
try:
    smi = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        text=True).strip()
    for line in smi.split("\n"):
        print(f"  GPU     : {line.strip()}")
except Exception:
    print("  GPU     : nvidia-smi not available")
try:
    import anthropic
    print(f"  anthropic  : {anthropic.__version__}")
except Exception:
    pass
print(f"  Model   : claude-opus-4-7")
print(f"  Chunks  : 10 frames/chunk  |  max_tokens : 4096")
print(f"  Retry   : 7 attempts, exponential back-off, cap 5 min")
print("=" * 64)


  NeurIPS Compute Summary  - 2026-04-28 17:05
  Python  : 3.10.11
  OS      : Windows-10-10.0.26100-SP0
  GPU     : NVIDIA H100 NVL, 95830 MiB
  GPU     : NVIDIA H100 NVL, 95830 MiB
  GPU     : NVIDIA H100 NVL, 95830 MiB
  GPU     : NVIDIA H100 NVL, 95830 MiB
  anthropic  : 0.96.0
  Model   : claude-opus-4-7
  Chunks  : 10 frames/chunk  |  max_tokens : 4096
  Retry   : 7 attempts, exponential back-off, cap 5 min


# Least-To-Most Prompting
**Least-To-Most Prompting** starts with simpler tasks and gradually increases complexity.

Key features:
- 8-step sequence escalating in complexity:
  - Low: object identification, people ID, location, basic actions
  - Medium: interactions, unusual behavior detection
  - High: criminal activity analysis, comprehensive timeline
- Each step builds context for the next
- Checkpoint/resume: saves progress after each video


In [4]:
import os
import json
import base64
import time
import random
import re
from collections import defaultdict
import anthropic


import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
# ================================================================
#  CONFIGURATION  -  edit these paths to match your machine
# ================================================================
FRAMES_DIR     = r"C:\Opeyemi\PROMPTS\FRAMES"   # pre-extracted frames
RESULTS_BASE   = r"C:\Opeyemi\PROMPTS\RESULTS"  # all JSON outputs
FRAME_EXT      = ".jpg"
FRAME_INTERVAL = 1    # 1=every frame; 2=every other; etc.
BATCH_SIZE     = 20   # max frames per API call (stay well under 100-image limit)
MAX_WORKERS    = 8    # parallel videos processed at once (tune to API rate-limit)

SAVE_DIR = r"C:\Opeyemi\PROMPTS\RESULTS\CLAUDE\LEAST-TO-MOST"
os.makedirs(SAVE_DIR, exist_ok=True)

CHECKPOINT_FILE = os.path.join(SAVE_DIR, "least_to_most_checkpoint.json")


# ================================================================
#  FRAME HELPERS
# ================================================================

def extract_frame_number(filename):
    """Return integer index from frame_00042.jpg style names."""
    name = os.path.splitext(filename)[0]
    m = re.search(r"frame[_\-]?(\d+)", name, re.IGNORECASE)
    if m:
        return int(m.group(1))
    nums = re.findall(r"\d+", name)
    return int(nums[-1]) if nums else 0


def discover_all_videos_and_frames(frames_dir=None):
    """
    Walk FRAMES_DIR and return a manifest of all extracted videos.

    Expected layout:
        FRAMES_DIR/
            Abuse/
                Abuse001_x264/
                    frame_00001.jpg ...
            Arrest/ ...

    Returns dict "<CrimeType>_<VideoStem>" -> {
        "crime_type", "video_id", "frames_dir", "frames"
    }
    """
    if frames_dir is None:
        frames_dir = FRAMES_DIR
    print("\n=== DISCOVERING FRAMES ===")
    print(f"    Root : {frames_dir}")
    all_videos = {}
    if not os.path.isdir(frames_dir):
        print(f"  ERROR: FRAMES_DIR not found: {frames_dir}")
        return all_videos
    crime_types = sorted([
        d for d in os.listdir(frames_dir)
        if os.path.isdir(os.path.join(frames_dir, d)) and not d.startswith("_")
    ])
    print(f"  Categories : {crime_types}")
    for crime_type in crime_types:
        cat_dir = os.path.join(frames_dir, crime_type)
        video_stems = sorted([
            d for d in os.listdir(cat_dir)
            if os.path.isdir(os.path.join(cat_dir, d))
        ])
        print(f"    {crime_type:20s}: {len(video_stems)} videos")
        for video_stem in video_stems:
            vdir = os.path.join(cat_dir, video_stem)
            frame_files = sorted(
                [ff for ff in os.listdir(vdir) if ff.lower().endswith(FRAME_EXT)],
                key=extract_frame_number,
            )
            if not frame_files:
                print(f"      WARNING: no {FRAME_EXT} frames in {vdir} - skipping")
                continue
            key = f"{crime_type}_{video_stem}"
            all_videos[key] = {
                "crime_type": crime_type,
                "video_id":   video_stem,
                "frames_dir": vdir,
                "frames":     frame_files,
            }
    print(f"  Total videos ready: {len(all_videos)}")
    return all_videos


def load_frames_for_video(video_info, frame_interval=1):
    """Read every frame_interval-th .jpg, base64-encode, return ordered list of (filename, b64)."""
    vdir        = video_info["frames_dir"]
    frame_files = video_info["frames"]
    video_id    = video_info["video_id"]
    selected    = frame_files[::frame_interval]
    label = "ALL" if frame_interval == 1 else f"every {frame_interval}th"
    print(f"  Loading {len(selected)} frames ({label}) for {video_id} ...")
    frames_list = []
    for ff in selected:
        fp = os.path.join(vdir, ff)
        try:
            with open(fp, "rb") as fh:
                b64 = base64.b64encode(fh.read()).decode("utf-8")
            frames_list.append((ff, b64))
        except Exception as e:
            print(f"    ERROR loading {ff}: {e}")
    print(f"  Loaded {len(frames_list)}/{len(selected)} frames OK")
    return frames_list   # list of (filename, b64_string)


# ================================================================
#  CHECKPOINT HELPERS
# ================================================================

def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        try:
            with open(CHECKPOINT_FILE, "r") as f:
                data = json.load(f)
            n = len(data.get("completed_videos", []))
            print(f"  Checkpoint: {n} videos already done - skipping them.")
            return data
        except Exception as e:
            print(f"  Could not read checkpoint ({e}) - starting fresh.")
    return {"completed_videos": [], "results": {}}


def save_checkpoint(data):
    os.makedirs(SAVE_DIR, exist_ok=True)
    tmp = CHECKPOINT_FILE + ".tmp"
    with open(tmp, "w") as f:
        json.dump(data, f, indent=2)
    os.replace(tmp, CHECKPOINT_FILE)


# ================================================================
#  CLAUDE API HELPER
# ================================================================

def make_claude_request_robust(client, model_name, messages,
                               system=None, temperature=0.1,
                               max_retries=7, base_wait=5):
    """Fault-tolerant Claude API call with exponential back-off."""
    RETRYABLE = (
        anthropic.RateLimitError,
        anthropic.APIConnectionError,
        anthropic.APITimeoutError,
    )
    PERMANENT = (
        anthropic.AuthenticationError,
        anthropic.PermissionDeniedError,
        anthropic.NotFoundError,
    )

    for attempt in range(1, max_retries + 1):
        try:
            kwargs = dict(
                model=model_name,
                max_tokens=4096,
                messages=messages,
            )
            if system:
                kwargs["system"] = system
            response = client.messages.create(**kwargs)
            return response.content[0].text

        except PERMANENT as e:
            msg = f"FATAL_ERROR: {type(e).__name__}: {e}"
            print(f"[FATAL] {msg}  -- will not retry.")
            return msg

        except RETRYABLE as e:
            wait = min(base_wait * (2 ** (attempt - 1)) + random.uniform(0, 2), 300)
            print(f"[Retry {attempt}/{max_retries}] {type(e).__name__}: {e}")
            print(f"  Waiting {wait:.1f}s before next attempt ...")
            time.sleep(wait)

        except anthropic.APIStatusError as e:
            if e.status_code >= 500:
                wait = min(base_wait * (2 ** (attempt - 1)) + random.uniform(0, 2), 300)
                print(f"[Retry {attempt}/{max_retries}] HTTP {e.status_code}: {e}")
                print(f"  Waiting {wait:.1f}s ...")
                time.sleep(wait)
            else:
                msg = f"FATAL_ERROR: HTTP_{e.status_code}: {e}"
                print(f"[FATAL] {msg}  -- will not retry.")
                return msg

        except Exception as e:
            if attempt < max_retries:
                print(f"[Retry {attempt}/{max_retries}] Unexpected {type(e).__name__}: {e}")
                time.sleep(base_wait * attempt)
            else:
                return f"ERROR: {type(e).__name__}: {e}"

    return f"ERROR: All {max_retries} attempts exhausted"


# ================================================================
#  LEAST-TO-MOST ANALYZER  (with batching)
# ================================================================

class LeastToMostAnalyzer:
    """
    Analyzes crime video frames using Least-to-Most prompting.

    The technique walks from simple low-complexity tasks to high-complexity
    reasoning, using earlier observations as scaffolding for later steps:

      Low complexity   1. List visible objects
                       2. Describe people present
                       3. Identify location / setting
                       4. Identify basic actions
      Medium           5. Analyze interactions
                       6. Detect unusual / suspicious behavior
      High             7. Analyze potential criminal activity
                       8. Build chronological timeline + classification
    """

    MODEL = "claude-opus-4-7"

    SYSTEM_PROMPT = (
        "You are an expert forensic video analyst specializing in crime detection "
        "and security surveillance. You analyze video frames methodically, noting "
        "details about people, actions, environment, and potential criminal activity. "
        "Be precise, objective, and thorough in your observations."
    )

    # Eight-step least-to-most prompt sequence
    PROMPT_SEQUENCE = [
        ("Low",
         "List all visible objects in these frames. Just identify what you can see "
         "(furniture, tools, vehicles, weapons, packages, etc.). Do not interpret yet."),
        ("Low",
         "Identify the people visible in these frames. How many are there? "
         "Describe each person's basic appearance: clothing, approximate age, "
         "general physical features. Track individuals across frames if possible."),
        ("Low",
         "Describe the location and setting shown in these frames. What kind of "
         "place is this (indoor/outdoor, residential/commercial/public)? Describe "
         "the spatial layout."),
        ("Low",
         "What basic actions are the people performing in these frames? List the "
         "simple, observable actions you see (walking, running, holding something, "
         "opening doors, etc.)."),
        ("Medium",
         "How are the people interacting with each other and with objects in the "
         "scene? Describe specific interactions in detail."),
        ("Medium",
         "Do you notice any unusual, concerning, or potentially suspicious "
         "behaviors in these frames? If so, what specifically seems unusual and why?"),
        ("High",
         "Based on your previous observations, analyze whether any potential "
         "criminal activities might be occurring. What specific elements suggest "
         "criminal behavior, and which alternative explanations are also plausible?"),
        ("High",
         "Using all your previous observations, construct a detailed chronological "
         "timeline of events shown in these frames. Then classify the activity as "
         "one of: Abuse, Arrest, Arson, Assault, Burglary, Explosion, Fighting, "
         "RoadAccidents, Robbery, Shooting, Shoplifting, Stealing, Vandalism, or "
         "Normal (no crime). Give a confidence level (0-100%) and the key "
         "supporting evidence."),
    ]

    def __init__(self, api_key: str, batch_size: int = BATCH_SIZE):
        self.client     = anthropic.Anthropic(api_key=api_key)
        self.batch_size = batch_size

    # ----------------------------------------------------------
    def _build_image_blocks(self, batch: list) -> list:
        """Convert a batch of (filename, b64) tuples into Anthropic content blocks."""
        blocks = []
        for fname, b64 in batch:
            blocks.append({
                "type": "image",
                "source": {
                    "type":       "base64",
                    "media_type": "image/jpeg",
                    "data":       b64,
                },
            })
            blocks.append({"type": "text", "text": f"[Frame: {fname}]"})
        return blocks

    # ----------------------------------------------------------
    def _run_step_on_batch(self, batch, batch_num, total_batches,
                           step_num, complexity, prompt_text, prior_steps_text):
        """Run a single LtM step on a single batch of frames, with prior context."""
        image_blocks = self._build_image_blocks(batch)
        framing = (
            f"You are on STEP {step_num}/{len(self.PROMPT_SEQUENCE)} of a "
            f"least-to-most analysis (complexity: {complexity}). "
            f"This is batch {batch_num}/{total_batches}.\n\n"
        )
        if prior_steps_text:
            framing += (
                "Previous step observations for THIS batch (build on them):\n"
                f"{prior_steps_text}\n\n"
            )
        framing += f"Step {step_num} task: {prompt_text}"

        messages = [{
            "role": "user",
            "content": image_blocks + [{"type": "text", "text": framing}],
        }]
        return make_claude_request_robust(
            self.client, self.MODEL, messages, system=self.SYSTEM_PROMPT)

    # ----------------------------------------------------------
    def _synthesize(self, per_batch_step_results, total_frames, total_batches):
        """Combine all batches' final-step answers into one classification."""
        joined = "\n\n".join(
            f"--- Batch {i+1} step-8 output ---\n{txt}"
            for i, txt in enumerate(per_batch_step_results)
        )
        synthesis_prompt = (
            f"You have just finished an 8-step least-to-most analysis across "
            f"{total_batches} batches covering {total_frames} frames of one "
            f"security video. Below is the final-step (timeline + classification) "
            f"output from each batch:\n\n"
            f"{joined}\n\n"
            f"Now produce the FINAL combined report:\n\n"
            f"1. SCENE DESCRIPTION: overall summary across all batches.\n"
            f"2. CRIME CLASSIFICATION: choose one of Abuse, Arrest, Arson, "
            f"Assault, Burglary, Explosion, Fighting, RoadAccidents, Robbery, "
            f"Shooting, Shoplifting, Stealing, Vandalism, or Normal.\n"
            f"3. CONFIDENCE: 0-100%.\n"
            f"4. KEY EVIDENCE: specific visual evidence supporting the choice.\n"
            f"5. RECOMMENDED ACTION: what law enforcement should do."
        )
        messages = [{"role": "user", "content": synthesis_prompt}]
        return make_claude_request_robust(
            self.client, self.MODEL, messages, system=self.SYSTEM_PROMPT)

    # ----------------------------------------------------------
    def analyze_frames(self, frames_list: list, video_id: str, crime_type: str) -> dict:
        """Full least-to-most analysis with batching."""
        total_frames  = len(frames_list)
        batches       = [
            frames_list[i:i + self.batch_size]
            for i in range(0, total_frames, self.batch_size)
        ]
        total_batches = len(batches)

        print(f"\n  [Least-to-Most] {video_id} | {total_frames} frames | "
              f"{total_batches} batches of up to {self.batch_size}")

        # per_batch_steps[batch_idx] = list of step responses for that batch
        per_batch_steps = [[] for _ in range(total_batches)]

        # Walk the 8 steps for every batch in turn
        for step_idx, (complexity, prompt_text) in enumerate(self.PROMPT_SEQUENCE, start=1):
            print(f"    Step {step_idx}/{len(self.PROMPT_SEQUENCE)} "
                  f"({complexity} complexity) ...")
            for b_idx, batch in enumerate(batches, start=1):
                # Build prior-steps summary for this batch
                prior = per_batch_steps[b_idx - 1]
                prior_text = "\n".join(
                    f"Step {i+1}: {txt}" for i, txt in enumerate(prior)
                ) if prior else ""
                resp = self._run_step_on_batch(
                    batch, b_idx, total_batches,
                    step_idx, complexity, prompt_text, prior_text,
                )
                per_batch_steps[b_idx - 1].append(resp)
                print(f"      Batch {b_idx}/{total_batches}: {len(resp)} chars")

        # Final synthesis using each batch's step-8 (final) output
        final_step_per_batch = [steps[-1] for steps in per_batch_steps]
        print(f"    Synthesizing {total_batches} batch finalizations ...")
        final_report = self._synthesize(final_step_per_batch, total_frames, total_batches)
        print(f"      Final report: {len(final_report)} chars")

        # Package per-batch step trace for full traceability
        batch_traces = []
        for b_idx, steps in enumerate(per_batch_steps, start=1):
            batch_traces.append({
                "batch_num": b_idx,
                "steps": [
                    {
                        "step":       i + 1,
                        "complexity": self.PROMPT_SEQUENCE[i][0],
                        "prompt":     self.PROMPT_SEQUENCE[i][1],
                        "response":   steps[i],
                    }
                    for i in range(len(steps))
                ],
            })

        return {
            "video_id":             video_id,
            "crime_type":           crime_type,
            "frames_analyzed":      total_frames,
            "total_batches":        total_batches,
            "batch_size":           self.batch_size,
            "prompting_technique":  "LEAST-TO-MOST",
            "model":                self.MODEL,
            "timestamp":            time.strftime("%Y-%m-%d %H:%M:%S"),
            "batch_traces":         batch_traces,
            "final_analysis":       final_report,
        }


# ================================================================
#  MAIN PIPELINE
# ================================================================

def process_all_crime_folders(api_key):
    """Analyse every video in FRAMES_DIR with checkpoint/resume support."""
    analyzer    = LeastToMostAnalyzer(api_key, batch_size=BATCH_SIZE)
    all_videos  = discover_all_videos_and_frames()
    if not all_videos:
        print("No videos found! Verify FRAMES_DIR path.")
        return {}

    cp          = load_checkpoint()
    all_results = cp.get("results", {})
    done_set    = set(cp.get("completed_videos", []))
    skipped     = []
    total       = len(all_videos)
    remaining   = {k: v for k, v in all_videos.items() if k not in done_set}

    print(f"\nVideos: total={total} | done={len(done_set)} | remaining={len(remaining)}")

    # ================================================================
    #  PARALLEL VIDEO PROCESSING
    # ================================================================
    checkpoint_lock = threading.Lock()
    print_lock      = threading.Lock()

    def _process_one_video(vkey, vinfo):
        """Process a single video in its own thread. Checkpoint writes are locked."""
        with print_lock:
            print(f"\n  [START] {vkey}")
        try:
            frames = load_frames_for_video(vinfo, FRAME_INTERVAL)
            if not frames:
                return vkey, None, "no frames"
            res = analyzer.analyze_frames(frames, vinfo["video_id"], vinfo["crime_type"])
            with checkpoint_lock:
                all_results[vkey] = res
                done_set.add(vkey)
                save_checkpoint({"completed_videos": list(done_set), "results": all_results})
            with print_lock:
                print(f"  [DONE]  {vkey}  ({len(done_set)}/{total})")
            return vkey, res, None
        except Exception as e:
            with print_lock:
                print(f"  [ERROR] {vkey}: {e}")
            with checkpoint_lock:
                save_checkpoint({"completed_videos": list(done_set), "results": all_results})
            return vkey, None, f"error: {e}"

    print(f"\n  Launching ThreadPoolExecutor with {MAX_WORKERS} parallel workers ...")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(_process_one_video, k, v) for k, v in remaining.items()]
        for fut in as_completed(futures):
            vkey, _, err = fut.result()
            if err:
                skipped.append(f"{vkey} ({err})")

    ts      = time.strftime("%Y%m%d_%H%M%S")
    summary = os.path.join(SAVE_DIR, f"least_to_most_summary_{ts}.json")
    with open(summary, "w") as f:
        json.dump(all_results, f, indent=2)

    if skipped:
        with open(os.path.join(SAVE_DIR, f"skipped_{ts}.txt"), "w") as f:
            f.write("\n".join(skipped))

    print(f"\nDone. Summary -> {summary}")
    print(f"  Processed: {len(all_results)} | Skipped: {len(skipped)}")
    return all_results


def run():
    """Entry point: load API key and kick off the full pipeline."""
    with open(r"C:\Opeyemi\PROMPTS\API-KEYS\claude.txt", "r") as _f:
        api_key = _f.read().strip()

    print("Least-to-Most Crime Video Analysis with Claude - Batched")
    print("=" * 60)
    print(f"  Frame interval : {FRAME_INTERVAL}  (1 = every frame)")
    print(f"  Batch size     : {BATCH_SIZE} frames per API call")
    print("=" * 60)

    try:
        results = process_all_crime_folders(api_key)
        print("\n" + "=" * 60)
        print(f"LEAST-TO-MOST COMPLETE! Videos processed: {len(results)}")
        print("=" * 60)
    except Exception as e:
        print(f"\nRun failed with error: {e}")
        raise


if __name__ == "__main__":
    run()


Least-to-Most Crime Video Analysis with Claude - Batched
  Frame interval : 1  (1 = every frame)
  Batch size     : 20 frames per API call

=== DISCOVERING FRAMES ===
    Root : C:\Opeyemi\PROMPTS\FRAMES
  Categories : ['Abuse', 'Assault', 'Burglary', 'Explosion', 'Fighting', 'RoadAccidents', 'Robbery', 'Shooting', 'Shoplifting', 'Stealing', 'Vandalism']
    Abuse               : 50 videos
    Assault             : 12 videos
    Burglary            : 100 videos
    Explosion           : 50 videos
    Fighting            : 50 videos
    RoadAccidents       : 150 videos
    Robbery             : 150 videos
    Shooting            : 50 videos
    Shoplifting         : 50 videos
    Stealing            : 100 videos
    Vandalism           : 50 videos
  Total videos ready: 812

Videos: total=812 | done=0 | remaining=812

  Launching ThreadPoolExecutor with 8 parallel workers ...

  [START] Abuse_Abuse001_x264
  Loading 91 frames (ALL) for Abuse001_x264 ...

  [START] Abuse_Abuse002_x264
  L